In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

data = pd.read_excel('Online Retail.xlsx')




In [ ]:
data.head(10)

In [ ]:
count = data['Description'].value_counts()
print(count)

In [ ]:
data.columns

In [ ]:
data.dtypes

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data = data.dropna(subset = ['CustomerID'])

In [ ]:
data.isnull().sum()

In [ ]:
data.shape

In [ ]:
data['CustomerID'].shape

In [ ]:
data.describe()

In [ ]:
price_max = data['UnitPrice'].max()
price_min = data['UnitPrice'].min()
price_quartail = data['UnitPrice'].quantile()
price_mean = data['UnitPrice'].mean()
price_med = data['UnitPrice'].median()
print(f'max={price_max}\nmin = {price_min}\nquartail={price_quartail}\nmean={price_mean}\nmedian={price_med}')

In [ ]:
price = data['UnitPrice']

In [ ]:
bottom = data['UnitPrice'].quantile(0.25)
top = data['UnitPrice'].quantile(0.75)
mid = data['UnitPrice'].quantile(0.50)
print(f'bottem = {bottom}\ntop={top}\nmid={mid}')

In [ ]:
iQR = top - bottom
lower_bound = bottom - 1.5*iQR
upper_bound = top +1.5*iQR
print(f'IQR = {iQR}\nlower_bound={lower_bound}\nupper_bound ={upper_bound}')

In [ ]:
for UnitPrice in data :
    if upper_bound > 7.5:
        print(upper_bound+1)



In [ ]:
count = 0
outlier_index = []

for i, value in enumerate(data['UnitPrice']):
    print(f"outlier at index{i}:{value}")
    count +=1
    outlier_index.append(i)

print("Total outlier:",count)

In [ ]:
data = data[(data['Quantity']>0) & (data['UnitPrice']>0)]
data.describe()

In [ ]:
mean = data['Quantity'].mean()
print(mean)

In [ ]:
avg =

In [ ]:
count = 0
outlier_index = []

for i, value in enumerate(data['UnitPrice']):
    print(f"outlier at index{i}:{value}")
    count +=1
    outlier_index.append(i)

print("Total outlier:",count)

In [ ]:
out = 406829 - 397884
print(out)

### Cohort Analysis

#### Invoice period: A string representation of the year and month of a single transaction/invoice.
#### Cohort group:A string representation of the the year and month of a customer's first purchase. This label is common across all invoices for a particular customer.
#### Cohort period / Cohort Index: A integer representation a customer's stage in its "lifetime". The number represents the number of months passed since the first purchase

In [ ]:
data.columns

In [ ]:
def get_month(x) :return dt.datetime(x.year,x.month,1)
data['InvoiceMonth'] = data['InvoiceDate'].apply(get_month)
grouping = data.groupby('CustomerID')['InvoiceMonth']
data['CohortMonth']=grouping.transform('min')
data.tail()

In [ ]:
def get_month_int(dframe,column):
    year = dframe[column].dt.year
    month = dframe[column].dt.month
    day= dframe[column].dt.day
    return year, month, day

invoice_year,invoice_month,_ = get_month_int(data,'InvoiceMonth')
cohort_year,cohort_month,_ = get_month_int(data,'CohortMonth')

year_diff = invoice_year - cohort_year
month_diff = invoice_month - cohort_month

data['CohortIndex'] = year_diff*12 + month_diff+1


In [ ]:
grouping = data.groupby(['CohortMonth','CohortIndex'])
cohort_data = grouping['CustomerID'].apply(pd.Series.nunique)

cohort_data = cohort_data.reset_index()
cohort_counts = cohort_data.pivot(index = 'CohortMonth',columns ='CohortIndex',values ='CustomerID')
cohort_counts

In [ ]:
##Retention table
cohort_size = cohort_counts.iloc[:,0]
retention = cohort_counts.divide(cohort_size,axis=0)
retention.round(3)*100

## Visualization

In [ ]:
plt.figure(figsize=(15,8))
plt.title('Retention rate')
sns.heatmap(data=retention,annot=True,fmt='.0%',vmin=0.0,vmax=0.5,cmap='BuPu_r')
plt.show()

In [ ]:
##average quantity for each cohort
grouping = data.groupby(['CohortMonth','CohortIndex'])
cohort_data = grouping['Quantity'].mean()
cohort_data = cohort_data.reset_index()
average_quantity = cohort_data.pivot(index='CohortMonth',columns='CohortIndex',values='Quantity')
average_quantity.round(1)
average_quantity.index = average_quantity.index.date

##Build the heatmap
plt.figure(figsize=(15,8))
plt.title('Average quantity for each cohort')
sns.heatmap(data=average_quantity,annot=True,vmin=0.0,vmax=20,cmap="BuGn_r")



In [ ]:
data['TotalSum'] = data['UnitPrice']*data['Quantity']
data['TotalSum']

print('Min Invoice Date:',data.InvoiceDate.dt.date.min(),'max Invoice Date:',
      data.InvoiceDate.dt.date.max())

data.head(3)

In [ ]:
snapshot_date = data['InvoiceDate'].max() + dt.timedelta(days=1)
snapshot_date

In [ ]:
data.columns

In [ ]:
rfm = data.groupby(['CustomerID']).agg({'InvoiceDate':lambda x :(snapshot_date -  x.max()).days,
                                        'InvoiceNo':'count','TotalSum':'sum'})

rfm.rename(columns={'InvoiceDate':'Recency','InvoiceNo':'Frequency','TotalSum':'MonetaryValue'}
           ,inplace=True)

rfm.head()